In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_4.py ──
"""
Shared infrastructure for MLFP02 Exercise 4 — A/B Testing & Experiment Design.

Contains: experiment data loading, two-arm extraction, power-analysis
primitives (z-values, required-n helper), random number generator factory,
and a small print helper used across the four technique files.

Technique-specific logic (power curves, SRM detection, Welch's t-test,
adaptive design) lives in the per-technique files — not here.
"""

import math
from pathlib import Path
from typing import NamedTuple

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# DESIGN CONSTANTS
# ════════════════════════════════════════════════════════════════════════

ALPHA: float = 0.05
POWER_TARGET: float = 0.80
DESIGN_MDE_PCT: float = 2.0  # relative MDE (percent of baseline mean)
SEED: int = 42

# Output directory for plots and comparison tables
OUTPUT_DIR = Path("outputs") / "mlfp02_ex4_experiment_design"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The experiment was DESIGNED with unequal allocation across four arms.
# Exercise 4 analyses control vs treatment_a, so its SRM check must use
# the designed control share within that pair (40 / (40 + 35)), never 50/50.
DESIGNED_ALLOCATION: dict[str, float] = {
    "control": 0.40,
    "treatment_a": 0.35,
    "treatment_b": 0.15,
    "variant_c": 0.10,
}
TREATMENT_ARM: str = "treatment_a"


# ════════════════════════════════════════════════════════════════════════
# DATA CONTAINER
# ════════════════════════════════════════════════════════════════════════


class TwoArmAB(NamedTuple):
    """Two-arm A/B subset pulled from the raw experiment frame."""

    experiment: pl.DataFrame  # full frame (all arms)
    ab_data: pl.DataFrame  # control + treatment_a only, sorted by `ts`
    ctrl_values: np.ndarray  # float64 numpy array, chronological order
    treat_values: np.ndarray  # float64 numpy array, chronological order
    n_control: int
    n_treatment: int
    n_total: int


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ════════════════════════════════════════════════════════════════════════


def load_experiment() -> TwoArmAB:
    """Load experiment_data.parquet and extract the two-arm A/B subset.

    Returns a TwoArmAB tuple with the raw frame, the filtered A/B frame,
    and numpy arrays for the control and treatment_a metric_value columns.
    The A/B frame gains a parsed datetime column `ts` and is sorted by it,
    so any "early vs late" comparison really is chronological (the raw
    file is NOT stored in time order).
    """
    loader = MLFPDataLoader()
    experiment = loader.load("mlfp02", "experiment_data.parquet")

    ab_data = (
        experiment.filter(
            pl.col("experiment_group").is_in(["control", TREATMENT_ARM])
        )
        .with_columns(pl.col("timestamp").str.to_datetime().alias("ts"))
        .sort("ts")
    )
    control = ab_data.filter(pl.col("experiment_group") == "control")
    treatment = ab_data.filter(pl.col("experiment_group") == TREATMENT_ARM)

    ctrl_values = control["metric_value"].to_numpy().astype(np.float64)
    treat_values = treatment["metric_value"].to_numpy().astype(np.float64)

    return TwoArmAB(
        experiment=experiment,
        ab_data=ab_data,
        ctrl_values=ctrl_values,
        treat_values=treat_values,
        n_control=control.height,
        n_treatment=treatment.height,
        n_total=ab_data.height,
    )


# ════════════════════════════════════════════════════════════════════════
# POWER-ANALYSIS PRIMITIVES
# ════════════════════════════════════════════════════════════════════════


def z_critical(
    alpha: float = ALPHA, power: float = POWER_TARGET
) -> tuple[float, float]:
    """Return (z_{alpha/2}, z_beta) — the two critical values used in sample-size formulas."""
    return stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)


def required_n_per_group(
    sigma: float,
    mde_absolute: float,
    alpha: float = ALPHA,
    power: float = POWER_TARGET,
) -> int:
    """Two-sample normal-approximation sample size per group.

    n = (z_{alpha/2} + z_beta)^2 * 2 * sigma^2 / delta^2

    Used by both the up-front design phase (Exercise 4.1) and the
    adaptive/sequential re-estimation phase (Exercise 4.4).
    """
    if mde_absolute <= 0:
        raise ValueError("mde_absolute must be positive")
    z_alpha_half, z_beta = z_critical(alpha, power)
    return math.ceil((z_alpha_half + z_beta) ** 2 * 2 * sigma**2 / mde_absolute**2)


def power_at_n(
    n_per_group: int,
    sigma: float,
    mde_absolute: float,
    alpha: float = ALPHA,
) -> float:
    """Compute achieved power for a given per-group sample size.

    Uses the non-central normal approximation:
        power = 1 - Phi(z_{alpha/2} - ncp) + Phi(-z_{alpha/2} - ncp)
    where ncp = delta / (sigma * sqrt(2/n)).
    """
    z_alpha_half, _ = z_critical(alpha)
    se = sigma * np.sqrt(2 / n_per_group)
    ncp = mde_absolute / se
    return float(
        1 - stats.norm.cdf(z_alpha_half - ncp) + stats.norm.cdf(-z_alpha_half - ncp)
    )


def cohens_d(delta: float, sigma: float) -> float:
    """Cohen's d effect size for two-sample means with pooled sigma."""
    return float(delta / sigma)


def interpret_cohens_d(abs_d: float) -> str:
    """Cohen's conventions: <0.2 negligible, <0.5 small, <0.8 medium, else large."""
    if abs_d < 0.2:
        return "negligible"
    if abs_d < 0.5:
        return "small"
    if abs_d < 0.8:
        return "medium"
    return "large"


# ════════════════════════════════════════════════════════════════════════
# SAMPLE RATIO MISMATCH
# ════════════════════════════════════════════════════════════════════════


def designed_control_share(arm: str = TREATMENT_ARM) -> float:
    """Designed share of control within the (control, arm) pair."""
    c = DESIGNED_ALLOCATION["control"]
    return c / (c + DESIGNED_ALLOCATION[arm])


def srm_chisquare(
    n_control: int, n_treatment: int, expected_control_share: float
) -> tuple[float, float]:
    """χ² SRM test of an observed two-arm split against its DESIGNED split.

    Returns (chi2, p_value). Use p < 0.01 as the SRM alarm. There is no
    50/50 default on purpose: the expected split comes from the design.
    """
    n_total = n_control + n_treatment
    expected = np.array(
        [n_total * expected_control_share, n_total * (1 - expected_control_share)]
    )
    chi2, p = stats.chisquare(np.array([n_control, n_treatment]), f_exp=expected)
    return float(chi2), float(p)


# ════════════════════════════════════════════════════════════════════════
# RANDOM NUMBER GENERATOR
# ════════════════════════════════════════════════════════════════════════


def make_rng(seed: int = SEED) -> np.random.Generator:
    """Factory for the per-exercise numpy Generator — deterministic across runs."""
    return np.random.default_rng(seed)


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_banner(title: str) -> None:
    """Print a uniform 70-char banner around a section title."""
    bar = "=" * 70
    print(f"\n{bar}\n  {title}\n{bar}")


def summarise_arm(name: str, values: np.ndarray) -> None:
    """Print a one-line summary of an experiment arm."""
    print(
        f"  {name:<10}: n={len(values):>6,}  "
        f"mean={values.mean():.4f}  std={values.std(ddof=1):.4f}"
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 4.4: Validity, Adaptive Design & Experiment Report
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement a data collection plan (Why/What/Where/How/Frequency)
#   - Evaluate experiment validity: SUTVA, interference, novelty effects
#   - Compute adaptive (sequential) sample-size re-estimation from a pilot
#   - Build a complete experiment analysis report with business decision
#   - Log the experiment report (design, validity, decision) to ExperimentTracker
#   - Connect adaptive design to capped regulatory-sandbox trials
#
# PREREQUISITES:
#   - MLFP02 Exercise 4.3 (Welch's t-test, confidence intervals)
#
# ESTIMATED TIME: ~45 minutes
#
# TASKS (5-phase R10):
#   1. Theory — SUTVA and why interference kills causal inference
#   2. Build — data collection plan + validity diagnostics
#   3. Train — adaptive sample-size re-estimation from pilot data
#   4. Visualise — full experiment report with business recommendation
#   5. Apply — adaptive design inside a regulatory sandbox
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import math

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from kailash_ml import ExperimentTracker
from scipy import stats



## THEORY — SUTVA and Interference


In [ ]:
# SUTVA (Stable Unit Treatment Value Assumption):
#   Each user's outcome depends ONLY on their own treatment assignment.
#
# Violations:
#   1. Network effects — treated users share with control users
#   2. Marketplace effects — treatment shifts supply/demand
#   3. Shared resources — treatment consumes server capacity
#
# ADAPTIVE DESIGN:
#   When variance is unknown upfront, start with a pilot phase to
#   estimate sigma, then compute the remaining sample size.



## TASK 1 — LOAD data


In [ ]:
print_banner("Exercise 4.4 — Validity, Adaptive Design & Report")

data: TwoArmAB = load_experiment()
rng = make_rng(SEED)

summarise_arm("Control", data.ctrl_values)
summarise_arm("Treatment", data.treat_values)

sigma_pooled = data.ctrl_values.std(ddof=1)
ctrl_mean = data.ctrl_values.mean()
mde_absolute = ctrl_mean * (DESIGN_MDE_PCT / 100)
n_required_per = required_n_per_group(sigma_pooled, mde_absolute, ALPHA, POWER_TARGET)



## TASK 2 — BUILD: Data Collection Plan & Validity Diagnostics


In [ ]:
print_banner("Data Collection Plan (Why/What/Where/How/Frequency)")

# TODO: Fill in the data collection plan dictionary. Each section is a
# dict of key-value pairs describing that aspect of data collection.
# The 5 sections are: WHY, WHAT, WHERE, HOW, FREQUENCY.
# Fill in the blank values. The success criteria must be checkable
# from the report (SRM, p-value, CI, lift vs the 2% MDE) — the decision
# code in Task 4 applies exactly these criteria.
plan = {
    "WHY (Hypotheses & Value)": {
        "Primary hypothesis": ____,
        "Secondary hypotheses": "Revenue impact, conversion impact",
        "Business value": "(illustrative) 1% engagement lift ~ $200K annual revenue",
        "Success criteria": ____,
    },
    "WHAT (Data Requirements)": {
        "Primary metric": "metric_value (per-user engagement/spend score)",
        "Secondary metrics": ____,
        "Covariates": "signup_date, device_type, country, prior_activity",
        "Guardrail metrics": "page_load_time, error_rate, support_tickets",
        "Minimum rows": f"{2 * n_required_per:,} (from power analysis)",
    },
    "WHERE (Data Sources)": {
        "Internal": "Event stream -> analytics warehouse, user_features table",
        "External": "None required for this experiment",
        "Schema": "user_id, timestamp, experiment_group, metric_value, revenue",
    },
    "HOW (Collection Method)": {
        "Assignment": "Server-side random hash on user_id (deterministic)",
        "Logging": "Event-sourced: every impression, click, purchase logged",
        "Quality": ____,
        "Privacy": "PII stripped at collection; analysis on anonymised IDs",
    },
    "FREQUENCY (Timing)": {
        "Collection frequency": "Real-time events, hourly batch aggregation",
        "Analysis frequency": ____,
        "Duration": f"~{2 * n_required_per // 5000} days at 5,000 users/day",
        "Stopping rule": "Analyse after target n reached; no early stopping",
    },
}

for section, items in plan.items():
    print(f"\n{section}")
    print("-" * 50)
    for key, value in items.items():
        print(f"  {key}: {value}")



### Checkpoint 1


In [ ]:
assert len(plan) == 5, "Plan must cover all 5 sections"
print("\n>>> Checkpoint 1 passed — data collection plan created\n")



### Validity Diagnostics


In [ ]:
print_banner("Experiment Validity Criteria")

# Check 1: Variance ratio (should be ~1 if no interference)
# TODO: Compute the variance ratio (treatment variance / control variance).
var_ratio = ____
print(f"  1. Variance ratio (treatment/control): {var_ratio:.3f}")
print(f"     Expected ~1.0 if no differential interference")
print(f"     Status: {'OK' if 0.8 < var_ratio < 1.2 else 'INVESTIGATE'}")

# Check 2: Distribution shape similarity (KS test)
# TODO: Run the two-sample KS test using stats.ks_2samp().
ks_stat, ks_p = ____
print(f"\n  2. KS test for distribution similarity: D={ks_stat:.4f}, p={ks_p:.6f}")
print(f"     A small p-value suggests distributions differ beyond just location shift")

# Check 3: Novelty effect — does the treatment EFFECT fade over time?
# Compare the lift (treatment - control) in the first half of the test
# window with the lift in the second half. Split by TIMESTAMP: the raw
# file is not stored in time order, so "first n rows" is not "early".
ab = data.ab_data  # sorted by the parsed `ts` column (see shared loader)
midpoint = ab["ts"].min() + (ab["ts"].max() - ab["ts"].min()) / 2
period_lifts = {}
for label, frame in [
    ("early", ab.filter(pl.col("ts") < midpoint)),
    ("late", ab.filter(pl.col("ts") >= midpoint)),
]:
    c = frame.filter(pl.col("experiment_group") == "control")["metric_value"].to_numpy()
    t = frame.filter(pl.col("experiment_group") != "control")["metric_value"].to_numpy()
    se = np.sqrt(c.var(ddof=1) / len(c) + t.var(ddof=1) / len(t))
    period_lifts[label] = (t.mean() - c.mean(), se)
(lift_early, se_early), (lift_late, se_late) = period_lifts["early"], period_lifts["late"]
# TODO: z-statistic for the difference between the early and late lifts.
# Hint: (lift_early - lift_late) / sqrt(se_early**2 + se_late**2)
novelty_z = ____
novelty_p = float(2 * stats.norm.sf(abs(novelty_z)))
novelty_detected = novelty_p < 0.05
print(
    f"\n  3. Novelty check (lift, early vs late half of the window): "
    f"early={lift_early:+.3f}, late={lift_late:+.3f}, z={novelty_z:.3f}, p={novelty_p:.4f}"
)
if novelty_detected:
    print("     WARNING: the lift changes over time — possible novelty/fatigue effect")
else:
    print("     OK: no evidence that the lift changes between the two halves")



### Checkpoint 2


In [ ]:
assert var_ratio > 0, "Variance ratio must be positive"
assert 0 <= ks_p <= 1, "KS p-value must be valid"
assert 0 <= novelty_p <= 1, "Novelty p-value must be valid"
print("\n>>> Checkpoint 2 passed — validity criteria assessed\n")



## TASK 3 — TRAIN: Adaptive Sample Size from Pilot


In [ ]:
print_banner("Adaptive Sample Size Calculation")

z_a, z_b = z_critical(ALPHA, POWER_TARGET)

# Simulate a pilot phase
pilot_n = 500
pilot_ctrl = rng.choice(data.ctrl_values, size=pilot_n, replace=True)
pilot_treat = rng.choice(data.treat_values, size=pilot_n, replace=True)

pilot_sigma = np.sqrt((pilot_ctrl.var(ddof=1) + pilot_treat.var(ddof=1)) / 2)
pilot_diff = pilot_treat.mean() - pilot_ctrl.mean()

# TODO: Re-compute required n based on pilot sigma estimate using
# required_n_per_group(pilot_sigma, mde_absolute, ALPHA, POWER_TARGET).
n_adaptive = ____

print(f"Pilot phase: n={pilot_n} per group")
print(f"Pilot sigma estimate: {pilot_sigma:.4f} (true sigma ~ {sigma_pooled:.4f})")
print(f"Pilot observed diff: {pilot_diff:+.4f}")
print(f"\nAdaptive required n per group: {n_adaptive:,}")
print(f"Original required n per group: {n_required_per:,}")
print(f"Ratio: {n_adaptive / n_required_per:.2f}x")
print(f"Remaining needed: {max(0, n_adaptive - pilot_n):,} per group")

# Multi-stage: how estimate improves with pilot size
print(f"\n--- Pilot Size vs Required n Stability ---")
for pilot_size in [100, 250, 500, 1000]:
    sigs = []
    for _ in range(100):
        pc = rng.choice(data.ctrl_values, size=pilot_size, replace=True)
        pt = rng.choice(data.treat_values, size=pilot_size, replace=True)
        s = np.sqrt((pc.var(ddof=1) + pt.var(ddof=1)) / 2)
        sigs.append(s)
    mean_sig = np.mean(sigs)
    std_sig = np.std(sigs)
    n_req = required_n_per_group(mean_sig, mde_absolute, ALPHA, POWER_TARGET)
    print(
        f"  Pilot n={pilot_size:>4}: sigma_hat={mean_sig:.4f} +/- {std_sig:.4f}, "
        f"required n={n_req:,}"
    )



### Checkpoint 3


In [ ]:
assert n_adaptive > 0, "Adaptive sample size must be positive"
print("\n>>> Checkpoint 3 passed — adaptive design completed\n")



## TASK 4 — VISUALISE: Full Experiment Report


In [ ]:
# Re-compute final statistics for the report
real_t_stat, real_p_val = stats.ttest_ind(
    data.treat_values, data.ctrl_values, equal_var=False
)
obs_diff = data.treat_values.mean() - data.ctrl_values.mean()
rel_lift = obs_diff / data.ctrl_values.mean() * 100

s1_sq_n1 = data.ctrl_values.var(ddof=1) / data.n_control
s2_sq_n2 = data.treat_values.var(ddof=1) / data.n_treatment
df_ws = (s1_sq_n1 + s2_sq_n2) ** 2 / (
    s1_sq_n1**2 / (data.n_control - 1) + s2_sq_n2**2 / (data.n_treatment - 1)
)
real_se = np.sqrt(s1_sq_n1 + s2_sq_n2)
t_crit = stats.t.ppf(1 - ALPHA / 2, df=df_ws)
welch_ci = (obs_diff - t_crit * real_se, obs_diff + t_crit * real_se)

pooled_std_real = np.sqrt(
    (data.ctrl_values.var(ddof=1) + data.treat_values.var(ddof=1)) / 2
)
cohens_d_real = obs_diff / pooled_std_real

# SRM check against the DESIGNED split (control 40 : treatment_a 35)
_, srm_p = srm_chisquare(data.n_control, data.n_treatment, designed_control_share())
srm_pass = srm_p >= 0.01
ci_excludes_zero = welch_ci[0] > 0 or welch_ci[1] < 0
meets_mde = obs_diff >= mde_absolute

# Adaptive sigma stability plot
fig_adapt = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Power Curve",
        "Pilot Size vs sigma Stability",
    ],
)

# Power curve subplot
sample_sizes = np.arange(500, n_required_per * 3, max(500, n_required_per // 20))
power_values = [
    power_at_n(int(ns), sigma_pooled, mde_absolute, ALPHA) for ns in sample_sizes
]
fig_adapt.add_trace(
    go.Scatter(
        x=sample_sizes.tolist(),
        y=power_values,
        mode="lines",
        name="Power",
        line={"color": "#2196F3"},
    ),
    row=1,
    col=1,
)
fig_adapt.add_hline(y=0.8, line_dash="dash", row=1, col=1)

# Sigma stability subplot
pilot_sizes_list = [50, 100, 200, 300, 500, 750, 1000]
sigma_means = []
sigma_stds = []
for ps in pilot_sizes_list:
    sigs = []
    for _ in range(200):
        pc = rng.choice(data.ctrl_values, size=ps, replace=True)
        pt = rng.choice(data.treat_values, size=ps, replace=True)
        sigs.append(np.sqrt((pc.var(ddof=1) + pt.var(ddof=1)) / 2))
    sigma_means.append(np.mean(sigs))
    sigma_stds.append(np.std(sigs))

fig_adapt.add_trace(
    go.Scatter(
        x=pilot_sizes_list,
        y=sigma_means,
        mode="lines+markers",
        name="sigma_hat",
        error_y={"type": "data", "array": sigma_stds, "visible": True},
        line={"color": "#FF9800"},
    ),
    row=1,
    col=2,
)
fig_adapt.add_hline(y=sigma_pooled, line_dash="dot", row=1, col=2)
fig_adapt.update_layout(
    title="Adaptive Design: Power Curve & Pilot Stability",
    height=400,
    template="plotly_white",
)
out_path = OUTPUT_DIR / "adaptive_design.html"
fig_adapt.write_html(str(out_path))
print(f"Saved: {out_path}")



### Checkpoint 4


In [ ]:
assert out_path.exists(), "Adaptive design plot must be saved"
print("\n>>> Checkpoint 4 passed — visualisations saved\n")



### Final business report


### Decision: apply the pre-registered success criteria, SRM first


Experiment: Recommendation Algorithm A/B Test
Duration: designed for {2 * n_required_per:,} total users
Actual: {data.n_total:,} users

SRM Check (vs designed split): p={srm_p:.4f} -> {'PASS' if srm_pass else 'FAIL — results may be biased'}

Primary Metric (metric_value):
  Control:   {data.ctrl_values.mean():.4f} +/- {data.ctrl_values.std():.4f}
  Treatment: {data.treat_values.mean():.4f} +/- {data.treat_values.std():.4f}
  Lift: {obs_diff:+.4f} ({rel_lift:+.2f}% relative; MDE {mde_absolute:.4f})
  p-value: {real_p_val:.6f}
  Cohen's d: {cohens_d_real:.4f}
  95% CI: [{welch_ci[0]:.4f}, {welch_ci[1]:.4f}]

Decision: {decision}

Validity: variance ratio {var_ratio:.2f}; novelty check p={novelty_p:.3f} -> {novelty_verdict}.


In [ ]:
if not srm_pass:
    decision = "DO NOT SHIP — investigate SRM before trusting any result"
elif real_p_val < ALPHA and ci_excludes_zero and meets_mde:
    decision = "SHIP — all pre-registered success criteria met"
elif real_p_val < ALPHA and obs_diff > 0:
    decision = "HOLD — significant, but the lift is below the pre-registered MDE"
elif real_p_val >= ALPHA and abs(obs_diff) > mde_absolute * 0.5:
    decision = "HOLD — more data needed"
else:
    decision = "NO SHIP — no meaningful effect detected"
if novelty_detected and decision.startswith("SHIP"):
    decision += " (monitor: lift changed over the window — possible novelty effect)"
novelty_verdict = (
    "possible novelty effect" if novelty_detected else "no novelty effect detected"
)

print_banner("EXPERIMENT REPORT")
print(
)
# INTERPRETATION: Cohen's d is small here even though the relative lift
# is large, because ~2% extreme values inflate the standard deviation.
# That is why the success criterion is stated as a lift vs the MDE the
# business pre-registered, not as a Cohen's d cut-off.



### Log the report to ExperimentTracker


In [ ]:
# The statistics are computed above; the tracker RECORDS the design, the
# validity checks and the decision so the experiment is auditable later.


async def log_experiment_report() -> str:
    store_url = "sqlite:///mlfp02_experiments.db"
    # TODO: Create the tracker against a local SQLite store.
    # Hint: await ExperimentTracker.create(store_url=store_url)
    tracker = ____
    try:
        async with tracker.track(
            experiment="mlfp02_ex4_ab_report", run_name="recommendation_ab"
        ) as run:
            await run.log_params(
                {
                    "treatment_arm": TREATMENT_ARM,
                    "design_mde_pct": str(DESIGN_MDE_PCT),
                    "alpha": str(ALPHA),
                    "power_target": str(POWER_TARGET),
                    "designed_control_share": f"{designed_control_share():.4f}",
                    "decision": decision,
                }
            )
            await run.log_metrics(
                {
                    "srm_p_value": float(srm_p),
                    "lift": float(obs_diff),
                    "relative_lift_pct": float(rel_lift),
                    "p_value": float(real_p_val),
                    "cohens_d": float(cohens_d_real),
                    "ci_lower": float(welch_ci[0]),
                    "ci_upper": float(welch_ci[1]),
                    "novelty_p_value": float(novelty_p),
                    "n_required_per_group": float(n_required_per),
                }
            )
            return run.run_id
    finally:
        await tracker.close()


report_run_id  = await log_experiment_report()
print(f"Logged experiment report to ExperimentTracker (run {report_run_id})")



### Checkpoint 5


In [ ]:
assert srm_pass or decision.startswith("DO NOT SHIP"), "Never ship on a failed SRM check"
assert report_run_id, "Report must be logged to ExperimentTracker"
print(">>> Checkpoint 5 passed — experiment report complete and logged\n")



## TASK 5 — APPLY: Adaptive Design Inside a Regulatory Sandbox


In [ ]:
# Singapore's financial regulator runs a FinTech Regulatory Sandbox in
# which firms trial new products with real customers for a limited
# period. Participants and time are capped, and the outcome variance is
# unknown before the trial starts — a natural fit for adaptive design.
#
# Scenario (illustrative numbers): a robo-advisor feature, outcome =
# annualised portfolio return (%), at most 2,000 participants.

print_banner("Applied — Regulatory Sandbox Adaptive Experiment")

sandbox_n_available = 2000
robo_sigma_guess = 8.0  # Cautious initial guess of return volatility (%)
robo_mde = 1.0  # Want to detect a 1 pp return improvement

# Phase 1: initial estimate from the guessed sigma
# TODO: Compute initial required n using the guessed sigma.
n_initial = ____
print(
    f"Initial estimate (sigma guess = {robo_sigma_guess}%): n={n_initial:,} per group"
)
feasible_initial = 2 * n_initial <= sandbox_n_available
print(
    f"Feasible in sandbox ({sandbox_n_available} available)? "
    f"{'YES' if feasible_initial else 'NO'}"
)

# Phase 2: a pilot estimates sigma from data (the scenario's true
# volatility, unknown to the team, is 4.2%)
pilot_n_sandbox = 200
true_robo_sigma = 4.2
pilot_returns = rng.normal(0.0, true_robo_sigma, size=(2, pilot_n_sandbox))
robo_sigma_pilot = float(np.sqrt(pilot_returns.var(axis=1, ddof=1).mean()))
# TODO: Compute revised required n with the pilot sigma estimate.
n_revised = ____
print(f"\nAfter pilot (n={pilot_n_sandbox} per arm, sigma_hat={robo_sigma_pilot:.2f}%):")
print(f"  Revised n per group: {n_revised:,}")
print(f"  Remaining: {max(0, n_revised - pilot_n_sandbox):,} per group")
feasible_revised = 2 * n_revised <= sandbox_n_available
print(f"  Feasible? {'YES' if feasible_revised else 'NO'}")

# Phase 3: power at the maximum available sample size
# TODO: Compute achieved power at the maximum available n per group.
achieved_power = ____
print(f"\nPower at maximum available n ({sandbox_n_available // 2} per group):")
print(f"  Achieved power: {achieved_power:.1%}")
print(
    f"  {'ADEQUATE' if achieved_power >= 0.8 else 'Consider larger MDE or extended sandbox'}"
)
if not feasible_initial and feasible_revised:
    print(
        "\nThe cautious sigma guess made the trial look infeasible; the pilot's\n"
        "variance estimate showed it fits within the sandbox's participant cap."
    )
elif feasible_initial:
    print("\nThe trial was feasible even under the initial guess; the pilot")
    print("confirms the plan and lets the team stop at the revised n.")
else:
    print("\nEven after the pilot the trial does not fit — widen the MDE,")
    print("extend the sandbox period, or redesign the outcome metric.")
# INTERPRETATION: When participants and time are capped, a pilot that
# re-estimates sigma lets you size the trial on evidence rather than a
# guess — avoiding both an underpowered trial and a wasted one.



### Checkpoint 6


In [ ]:
assert n_revised > 0, "Revised n must be positive"
assert 0 < achieved_power <= 1, "Achieved power must be valid"
assert not feasible_initial, "With sigma guess 8%, the initial design should not fit"
print("\n>>> Checkpoint 6 passed — sandbox scenario completed\n")



## REFLECTION


- Data collection plan: Why/What/Where/How/Frequency framework
  - SUTVA: interference, novelty effects, variance-ratio checks
  - Adaptive design: pilot -> estimate sigma -> compute remaining n
  - Pilot stability: larger pilots => more reliable sigma estimates
  - Complete experiment report: SRM gate first, then pre-registered
    success criteria; logged to ExperimentTracker
  - Novelty check: compare the lift across time, split by timestamp
  - Applied: adaptive design inside a capped regulatory sandbox

  NEXT: In Exercise 5 you'll build linear regression from scratch.
  You'll derive OLS using matrix algebra, test coefficient significance
  with t-statistics, detect multicollinearity, and run residual
  diagnostics — all on HDB price prediction data.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print(">>> Exercise 4.4 complete — Validity, Adaptive Design & Report")
print("\n>>> Exercise 4 complete — A/B Testing and Experiment Design")

